# Xarray-Spatial I/O: JPEG 2000 compression

Satellite imagery (Sentinel-2, Landsat) often ships in JPEG 2000 format. The `geotiff` module lets you write and read J2K-compressed GeoTIFFs on CPU (via glymur/OpenJPEG) or GPU (via nvJPEG2000).

### What you'll build

1. [Generate synthetic elevation data](#data)
2. [Write with JPEG 2000 lossless compression](#write-j2k)
3. [Read back and verify the roundtrip](#read-verify)
4. [Compress a multi-band RGB image](#multi-band)
5. [GPU acceleration notes](#gpu)

![preview](images/40_jpeg2000_compression.png)

Import the libraries we need.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import tempfile
import os

from xrspatial.geotiff import open_geotiff, to_geotiff

<a id="data"></a>

## Data

A small 256x256 synthetic terrain raster, stored as uint16 to mimic satellite imagery.

In [ ]:
# Create a 256x256 synthetic terrain (uint16, typical for satellite imagery)
rng = np.random.RandomState(42)
yy, xx = np.meshgrid(np.linspace(-2, 2, 256), np.linspace(-2, 2, 256), indexing='ij')
terrain = np.exp(-(xx**2 + yy**2)) * 10000 + rng.normal(0, 100, (256, 256))
terrain = np.clip(terrain, 0, 65535).astype(np.uint16)

da = xr.DataArray(
    terrain,
    dims=['y', 'x'],
    coords={
        'y': np.linspace(45.0, 44.0, 256),
        'x': np.linspace(-120.0, -119.0, 256),
    },
    attrs={'crs': 4326},
    name='elevation',
)

A Gaussian peak with random noise, clipped to the uint16 range (0 to 65535). The coordinate system is EPSG:4326.

In [ ]:
da.plot.imshow(cmap='terrain', figsize=(6, 5))
plt.title('Synthetic elevation (uint16)')
plt.tight_layout()

<a id="write-j2k"></a>

## Write with JPEG 2000 (lossless)

Pass `compression='jpeg2000'` to `to_geotiff`. The default is lossless encoding.

In [ ]:
tmpdir = tempfile.mkdtemp(prefix='j2k_demo_')

# Write with JPEG 2000 compression
j2k_path = os.path.join(tmpdir, 'elevation_j2k.tif')
to_geotiff(da, j2k_path, compression='jpeg2000')

# Compare file sizes with deflate
deflate_path = os.path.join(tmpdir, 'elevation_deflate.tif')
to_geotiff(da, deflate_path, compression='deflate')

none_path = os.path.join(tmpdir, 'elevation_none.tif')
to_geotiff(da, none_path, compression='none')

j2k_size = os.path.getsize(j2k_path)
deflate_size = os.path.getsize(deflate_path)
none_size = os.path.getsize(none_path)

print(f"Uncompressed:  {none_size:>8,} bytes")
print(f"Deflate:       {deflate_size:>8,} bytes  ({deflate_size/none_size:.1%} of original)")
print(f"JPEG 2000:     {j2k_size:>8,} bytes  ({j2k_size/none_size:.1%} of original)")

<a id="read-verify"></a>

## Read back and verify lossless roundtrip

`open_geotiff` auto-detects the compression from the TIFF header. No special arguments needed.

In [ ]:
# Read back and check lossless roundtrip
da_read = open_geotiff(j2k_path)

print(f"Shape: {da_read.shape}")
print(f"Dtype: {da_read.dtype}")
print(f"CRS:   {da_read.attrs.get('crs')}")
print(f"Exact match: {np.array_equal(da_read.values, terrain)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
da.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=False)
axes[0].set_title('Original')
da_read.plot.imshow(ax=axes[1], cmap='terrain', add_colorbar=False)
axes[1].set_title('After JPEG 2000 roundtrip')
plt.tight_layout()

<a id="multi-band"></a>

## Multi-band example (RGB)

JPEG 2000 also handles multi-band imagery, which is the common case for satellite data.

In [ ]:
# Create a 3-band uint8 image (blue + orange gradient, avoids red/green pairing)
rgb = np.zeros((128, 128, 3), dtype=np.uint8)
rgb[:, :, 0] = np.linspace(0, 255, 128).astype(np.uint8)[None, :]  # red gradient
rgb[:, :, 2] = np.linspace(0, 255, 128).astype(np.uint8)[:, None]  # blue gradient
rgb[:, :, 1] = 80  # muted green channel

da_rgb = xr.DataArray(
    rgb, dims=['y', 'x', 'band'],
    coords={'y': np.arange(128), 'x': np.arange(128), 'band': [0, 1, 2]},
)

rgb_path = os.path.join(tmpdir, 'rgb_j2k.tif')
to_geotiff(da_rgb, rgb_path, compression='jpeg2000')

da_rgb_read = open_geotiff(rgb_path)
print(f"RGB shape: {da_rgb_read.shape}, dtype: {da_rgb_read.dtype}")
print(f"Exact match: {np.array_equal(da_rgb_read.values, rgb)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(rgb)
axes[0].set_title('Original RGB')
axes[1].imshow(da_rgb_read.values)
axes[1].set_title('After J2K roundtrip')
plt.tight_layout()

<a id="gpu"></a>

## GPU acceleration

On systems with nvJPEG2000 installed (CUDA toolkit, RAPIDS environments), pass `gpu=True` to use GPU-accelerated J2K encode/decode. The API is the same; it falls back to CPU if the library isn't found.

```python
# GPU write (nvJPEG2000 if available, else CPU fallback)
to_geotiff(cupy_data, "output.tif", compression="jpeg2000", gpu=True)

# GPU read (nvJPEG2000 decode if available)
da = open_geotiff("satellite.tif", gpu=True)
```

In [ ]:
# Cleanup temp files
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)

### References

- [JPEG 2000 (Wikipedia)](https://en.wikipedia.org/wiki/JPEG_2000)
- [Glymur: Python interface to OpenJPEG](https://glymur.readthedocs.io/)
- [OGC GeoTIFF standard](https://www.ogc.org/standard/geotiff/)
- [nvJPEG2000 (NVIDIA)](https://developer.nvidia.com/nvjpeg)
- [xarray-spatial GeoTIFF API docs](https://xarray-spatial.readthedocs.io/)